### Middleware :: 
(Check, modify, or control execution)

Middleware provides a way to more tightly control what happens inside the agent.Middleware is useful for the following:

- Tracking agent behavior with logging,analytics,and debugging.   
- Transforming prompts,tool selection,and output formatting.    
- Adding retries,fallbacks and early termination logic.    
- Applying rate limits, guardrails and PII detection






### Some BUILT IN MIDDLEWARE ::

1.SUMMARIZATION : Summarizes older messages .   
2.HUMAN IN THE LOOP : Pauses for human approval .   
3.MODEL CALL LIMIT etc.....

In [32]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

### Summarization MiddleWare
Summarizes older conversation messages to reduce the amount of history sent to the model. Summarization is useful for the following :
- Long-running conversation that exceed context windows.  
- Multi-turn dialogues with extension history . 
- Applications where preserving full conversation context matters.

#### Based on length of the msg

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
### MessageBased Summarrization
agent=create_agent(
    model="google_genai:gemini-3.5-flash",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}


In [34]:
question=[
    "What is 8*65?",
    "What do you mean by NLP?",
    "What is 23*6 ?",
    "What is the full form of CDS ?",
    "What is the square of 23 ?",
    "Explain the lifeCycle of Software Development "
]
for q in question:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(f"messages:{response}")
    print(f"Length of the reponse {len(response['messages'])}")

messages:{'messages': [HumanMessage(content='What is 8*65?', additional_kwargs={}, response_metadata={}, id='bed63704-41ac-4eec-a399-47a6d0f5f654'), AIMessage(content=[{'type': 'text', 'text': '8 * 65 = **520**', 'extras': {'signature': 'ErMECrAEAWkUfRNpJdFc+B2KMDgV66aTkyzdaFV1E/fjuyf09iWdnRQK0fwn3IUd7mFfOmqcwuYx4dZfmfWxsvpJGGjEcibNA4GxaHWbYgJAmONR4quIbjcsmHV0765yO96vflWJWSaGyAhM4vPxbZx7T3JeJRLC39KYBiUc0/Rt3+lXWewfiSJnTecnkOqIjkZbPWBd89LwX+K3CGC8Gntu6lxC2HGSgM90KSFJYuLw+/uTibTcFWfZm84CNzeb9bwzCDDGlkm5Til5CdppRaLItIfyD6zpV9o373cg+3eqg88A82dU62iDoylL0EvWuqRTeRlLRuLyzQDPlN9fu2OioDBBzdeWewxrgy+hhWtkvs0Ao+xOjV7e+fVkODQKuk2MfMZU6EpdUJ4BYaAHyBkcJZoZcm1bRdjNC/A/zHWb/o3+C0Udmh7w3U2MxNDsrwGcILlhYf+FsWds7bpQZn2+7aiqZ3SbSj5C8E9XYAPKJR7YUDb66ubRowYtQVsteNxUnH6wxYKemXXMag3j696juQD05LbtWlcGYEkjVpFT5paJ5vcu7dpcW5G2RriBiA0K2Cl5udBDLcZqSkC1WFaqYPoKrNdtp0yLsw9nzddamauRWDbAEfdbJp/mnYfpBaQk0U5XuLcdvB5U9tkMSaBwa2BC/yV9rXuDHUSClI+emIsPb38jEo7wYriwjRQdEQ5Jkkuf+B55eKJEXIUvx4tLZ/9wjOeRLcZiarA+jmHUsv0x4hk='}}], 

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 34.600370192s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Gen

#### Based on token size

In [35]:
### Based on token size
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star, $350/night,spa,pool,gym
    2.City Inn - 4 star,$180/night,business center
    3.Budget stay - 3 star,$75/night,free wifi """
### MessageBased Summarrization
agent=create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.8-flash",
            trigger=("tokens",650),
            keep=("tokens",200)
        )
    ]
)
config={"configurable":{"thread_id":"test-1"}}
#Token counter (approximate)
def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars//4   #4 Char =1 Token

In [ ]:
#Run test
cities=["Nainital","Manali","New York","Dubai"]
for city in cities:
    response=agent.invoke({"messages":[HumanMessage(content=f"Find hotel in {city}")]},config=config)
    token=count_tokens(response["messages"])
    print(f"{city} :{token} tokens ,{len(response['messages'])} messages")
    

Nainital :277 tokens ,4 messages
Manali :550 tokens ,8 messages


Based on Fraction

In [1]:
### Based on token size
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1.Grand Hotel - 5 star, $350/night,spa,pool,gym
    2.City Inn - 4 star,$180/night,business center
    3.Budget stay - 3 star,$75/night,free wifi """
### MessageBased Summarrization
agent=create_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.8-flash",
            trigger=("fraction",0.005),#0.5%= 640 tokens (aproximate)
            keep=("fraction",0.002)#0.2% =256 tokens
        )
    ]
)
config={"configurable":{"thread_id":"test-1"}}
#Token counter (approximate)
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages)//4
    

In [2]:
#test
cities=["Nainital","Manali","New York","Dubai"]
for city in cities:
    response=agent.invoke(
        { "messages":[HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens=count_tokens(response["messages"])
    fraction=token/128000
    print(f"{city} :{tokens} token({fraction:.4%}),len(response['messages']) msgs")

NameError: name 'token' is not defined

### Human In the Loop MiddleWare

Pause agent execution for human approval,editing,or rejection of tool calls before they execute. Human-in-the-loopis useful for the following:
- High stake operations requiring human approval(e.g.database writes,financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guide the agent.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """ Mock Function to read the email by its ID."""
    return f"Email content for ID:{email_id}"
def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to write the email."""
    return f"Email sent to {recipient} with subject '{subject}'"


In [12]:
agent=create_agent(
    model="google_genai:gemini-3.8-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                send_email_tool:{
                    "allowed_decisions":["approve","edit","reject"]
                },
                read_email_tool:False
            }
        )
    ]
)

In [13]:
config={"configurable":{"thread_id":"test-approve"}}
#request
result=agent.invoke(
    {"messages":[HumanMessage(content="Send email to john@gmail.com with subject 'Hello' and 'How are you ?' ")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@gmail.com with subject 'Hello' and 'How are you ?' ", additional_kwargs={}, response_metadata={}, id='b239ffc5-223e-450c-9e25-4699a666e652'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you ?", "recipient": "john@gmail.com", "subject": "Hello"}'}, '__gemini_function_call_thought_signatures__': {'call_559143': 'EpcDCpQDAWkUfRMraqV11AP20wDX+cGqdFLJWwTsuQltT2n4M3rulXI4Y3TzBBmjk81eHVGAZGSM5qyulqI+82CeVCfyhjI8nW4TvBOMu0UY2bjIsjoRdeiMVtA+/Is8zWia00dpA1o5kqdHf9HXCOuq3c2XbMdYE1SwZYLZk8w39FnRu2Gadne3Jwvni4mSRwVzniwXdf0+oJb2lkgSeswxGyFV/+hdc07pcezUTArQpK2g/qYNSpnwpvkGeQQTVvK+OEZz7nT3YiwuSDxePremyMBgS8Ag14g55ud3EELeKhMjv6poPdJ2y8YYmYCh/xpbrc2rihmKAxMTan6RlP6YPE6oKI84aU3yMmK89ufl/9Hun/VgqKdtkO8QZXbRJ2dYeinqHsXaSfMwav9hDoTAUsJGloCySGDO3xJW56OJobg6iY/MTeeM2f2p0abAURpMztFMhVjrL2PfmdEHHXoL6fNELbK2w7GVJ/tukocH7NlUDOb6Qj2aXbVAUYEMhj9KlcrJdOx/NI+zVCyoXO9OvlD1xMDPbTA='}}, resp

In [14]:
#Approve
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused ! Approving...")
    result=agent.invoke(
        command={
            "decisions":[
                {"type":"approve"}
            ]
        },
        config=config
    )
    print(f"Result :{result['messages'][-1].content}")
    

### Reject

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str)->str:
    """ Mock Function to read the email by its ID."""
    return f"Email content for ID:{email_id}"
def send_email_tool(recipient:str,subject:str,body:str)->str:
    """Mock function to write the email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent=create_agent(
    model="google_genai:gemini-3.8-flash",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                send_email_tool:{
                    "allowed_decisions":["approve","edit","reject"]
                },
                read_email_tool:False
            }
        )
    ]
)


In [17]:
config={"configurable":{"thread_id":"test-reject"}}
#requst
result=agent.invoke({
    "messages":[HumanMessage(content="Send email to john@test.com with subject 'hello' and body 'how are you '")]},
    config=config
)
        

In [18]:
#reject
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused ! Approving...")
    result=agent.invoke(
        command={
            "decisions":[
                {"type":"reject"}
            ]
        },
        config=config
    )
    print(f"Result :{result['messages'][-1].content}")
    

### Editing

In [19]:
#editing
from langgraph.types import Command
if "__interrupt__" in result:
    print("Paused ! Editing...")
    result=agent.invoke(
        command={
            "decisions":[
                {"type":"reject",
                 "edited_action":{
                     "name":"send_email-tool",
                     "args":{
                         "recipient":"correct@email.com",
                         "subject":"Corrected Subject",
                         "body":"This was edited by human before sending."
                     }
                 }
                }
            ]
        },
        config=config
    )
    print(f"Result :{result['messages'][-1].content}")
    